## Formulação do MDP

Este notebook implementa o jogo da velha como um Processo de Decisão de Markov (MDP). O estado (`S`) é o tabuleiro 3x3 representado por um vetor de 9 posições, usando `1` para o agente, `-1` para o oponente e `0` para casas vazias. As ações (`A`) são os índices das casas vazias disponíveis no tabuleiro. A função de recompensa (`R`) retorna `+1` quando o agente vence, `-1` quando o oponente vence e `0` em empates ou passos não terminais. A dinâmica de transição (`P`) aplica primeiro a jogada do agente e, se o jogo ainda não terminou, simula uma jogada aleatória válida do oponente.

In [3]:
import random
import subprocess
import sys
import numpy as numpy


class TicTacToeEnv:
    """
    Ambiente de Reinforcement Learning para o jogo da velha, implementado do zero.

    Este ambiente modela formalmente um MDP:
    - S: estados possíveis do tabuleiro.
    - A: ações válidas em cada estado.
    - R: recompensa após cada transição.
    - P: dinâmica de transição entre estados.
    """

    AGENT = 1
    OPPONENT = -1
    EMPTY = 0

    WINNING_LINES = (
        (0, 1, 2),  # linha superior
        (3, 4, 5),  # linha do meio
        (6, 7, 8),  # linha inferior
        (0, 3, 6),  # coluna esquerda
        (1, 4, 7),  # coluna do meio
        (2, 5, 8),  # coluna direita
        (0, 4, 8),  # diagonal principal
        (2, 4, 6),  # diagonal secundária
    )

    def __init__(self):
        # Estado inicial do MDP: tabuleiro vazio com 9 posições.
        self.board = np.zeros(9, dtype=int)
        self.done = False

    def reset(self):
        """
        Reinicia o ambiente para o estado inicial.

        Returns:
            Cópia do tabuleiro vazio, ou seja, o estado inicial S0.
        """
        self.board = np.zeros(9, dtype=int)
        self.done = False
        return self.board.copy()

    def get_state_hash(self):
        """
        Retorna uma representação imutável do estado atual.

        Esse método é importante para Q-learning tabular, pois permite usar
        o estado como chave de dicionário nas atualizações de Bellman.
        """
        return tuple(self.board.tolist())

    def get_available_actions(self):
        """
        Componente A do MDP: retorna as ações válidas no estado atual.

        Cada ação é um índice entre 0 e 8 que representa uma casa vazia.
        """
        return [index for index, value in enumerate(self.board) if value == self.EMPTY]

    def check_winner(self):
        """
        Verifica se algum jogador atingiu uma condição terminal de vitória.

        Returns:
            1 se o agente venceu.
            -1 se o oponente venceu.
            0 se ainda não há vencedor.
        """
        for line in self.WINNING_LINES:
            line_sum = int(sum(self.board[index] for index in line))

            if line_sum == 3 * self.AGENT:
                return self.AGENT

            if line_sum == 3 * self.OPPONENT:
                return self.OPPONENT

        return self.EMPTY

    def _is_draw(self):
        """
        Um empate ocorre quando não há vencedor e não existem ações válidas.
        """
        return self.check_winner() == self.EMPTY and len(self.get_available_actions()) == 0

    def step(self, action):
        """
        Componente P do MDP: executa uma transição a partir da ação do agente.

        Sequência da transição:
        1. O agente marca a posição escolhida com 1.
        2. O ambiente verifica recompensa terminal para vitória ou empate.
        3. Se o jogo continua, o oponente escolhe uma ação válida aleatória.
        4. O ambiente verifica novamente vitória, empate ou passo não terminal.

        Args:
            action: índice inteiro entre 0 e 8 escolhido pelo agente.

        Returns:
            next_state: cópia do novo estado do tabuleiro.
            reward: recompensa R da transição.
            done: indica se o episódio terminou.
        """
        if self.done:
            raise RuntimeError("O episódio terminou. Chame reset() antes de executar outro step().")

        if action not in range(9):
            raise ValueError("A ação deve ser um índice inteiro entre 0 e 8.")

        if self.board[action] != self.EMPTY:
            raise ValueError("Ação inválida: a casa escolhida já está ocupada.")

        # Transição do agente: aplica a ação escolhida no estado atual.
        self.board[action] = self.AGENT

        # Componente R: recompensa terminal positiva se o agente venceu.
        if self.check_winner() == self.AGENT:
            self.done = True
            return self.board.copy(), 1, self.done

        # Componente R: empate gera recompensa 0 e encerra o episódio.
        if self._is_draw():
            self.done = True
            return self.board.copy(), 0, self.done

        # Dinâmica do ambiente: o oponente faz uma jogada aleatória válida.
        opponent_action = random.choice(self.get_available_actions())
        self.board[opponent_action] = self.OPPONENT

        # Componente R: recompensa terminal negativa se o oponente venceu.
        if self.check_winner() == self.OPPONENT:
            self.done = True
            return self.board.copy(), -1, self.done

        # Componente R: empate após a jogada do oponente também retorna 0.
        if self._is_draw():
            self.done = True
            return self.board.copy(), 0, self.done

        # Passo não terminal: recompensa 0 e o episódio continua.
        return self.board.copy(), 0, self.done

    def render(self):
        """
        Visualiza o estado atual do tabuleiro no terminal/notebook.
        """
        symbols = {
            self.AGENT: "X",
            self.OPPONENT: "O",
            self.EMPTY: " ",
        }

        for row in range(3):
            start = row * 3
            cells = [symbols[int(value)] for value in self.board[start:start + 3]]
            print(f" {cells[0]} | {cells[1]} | {cells[2]} ")

            if row < 2:
                print("---+---+---")


## Sanity check

O laço abaixo executa um episódio completo usando ações aleatórias para o agente. A cada iteração, selecionamos uma ação válida, aplicamos `step(action)` e deixamos o ambiente responder com a jogada aleatória do oponente. O objetivo do teste é verificar se o ambiente inicializa corretamente, respeita ações válidas, avança as transições do MDP, detecta vitória/empate e retorna uma recompensa final coerente.

In [4]:
random.seed(42)

env = TicTacToeEnv()
state = env.reset()
done = False
final_reward = 0
step_count = 0

while not done:
    # O agente aleatório escolhe apenas ações disponíveis no estado atual.
    action = random.choice(env.get_available_actions())

    # O ambiente aplica a ação do agente, simula o oponente e retorna S', R e done.
    state, final_reward, done = env.step(action)
    step_count += 1

print("Tabuleiro final:")
env.render()
print(f"\nHash do estado final: {env.get_state_hash()}")
print(f"Quantidade de passos do agente: {step_count}")
print(f"Recompensa final: {final_reward}")


NameError: name 'np' is not defined